# Data Preprocessing Pipeline: Latest Data Science Job Salaries (2020–2025)

This notebook transforms raw JSON data into a structured and normalized format.

**Input:** Raw JSON dataset  
**Processing:** Type casting, column standardization, missing value handling, validation  
**Output:** Cleaned JSON dataset prepared for database upload

In [3]:
# Importing libraries
import pandas as pd
import matplotlib.pyplot as plt
import pycountry

In [1]:
# Helper function to transform ISO2 country names to ISO3 (standard for all datasets in our project)
def to_iso3(code):
    try:
        return pycountry.countries.get(alpha_2=code).alpha_3
    except:
        return None

In [5]:
df = pd.read_json('raw_data\DataScience_salaries_2025_raw.json') # Check the directory where raw data file is stored

<>:1: SyntaxWarning: invalid escape sequence '\D'
<>:1: SyntaxWarning: invalid escape sequence '\D'
C:\Users\Honor\AppData\Local\Temp\ipykernel_11600\4168381544.py:1: SyntaxWarning: invalid escape sequence '\D'
  df = pd.read_json('raw_data\DataScience_salaries_2025_raw.json') # Check the directory where raw data file is stored


In [7]:
# glancing at the data
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size
0,2025,MI,FT,Research Scientist,208000,USD,208000,US,0,US,M
1,2025,MI,FT,Research Scientist,147000,USD,147000,US,0,US,M
2,2025,SE,FT,Research Scientist,173000,USD,173000,US,0,US,M
3,2025,SE,FT,Research Scientist,117000,USD,117000,US,0,US,M
4,2025,MI,FT,AI Engineer,100000,USD,100000,US,100,US,M


In [9]:
# for setting upper and lower bound
print(df['work_year'].min())
print(df['work_year'].max())

2020
2025


In [11]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93597 entries, 0 to 93596
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   work_year           93597 non-null  int64 
 1   experience_level    93597 non-null  object
 2   employment_type     93597 non-null  object
 3   job_title           93597 non-null  object
 4   salary              93597 non-null  int64 
 5   salary_currency     93597 non-null  object
 6   salary_in_usd       93597 non-null  int64 
 7   employee_residence  93597 non-null  object
 8   remote_ratio        93597 non-null  int64 
 9   company_location    93597 non-null  object
 10  company_size        93597 non-null  object
dtypes: int64(4), object(7)
memory usage: 7.9+ MB


In [13]:
df.isna().sum()

work_year             0
experience_level      0
employment_type       0
job_title             0
salary                0
salary_currency       0
salary_in_usd         0
employee_residence    0
remote_ratio          0
company_location      0
company_size          0
dtype: int64

In [15]:
df.describe()

,work_year,salary,salary_in_usd,remote_ratio
count,93597.000000,9.359700e+04,93597.000000,93597.000000
mean,2024.086434,1.623541e+05,157547.696774,21.455816
std,0.641449,2.221425e+05,73649.113729,40.954704
min,2020.000000,1.400000e+04,15000.000000,0.000000
25%,2024.000000,1.062600e+05,106250.000000,0.000000
50%,2024.000000,1.470000e+05,146232.000000,0.000000
75%,2024.000000,1.990000e+05,198000.000000,0.000000
max,2025.000000,3.040000e+07,800000.000000,100.000000


## Typecasting

In [17]:
# Casting object datatypes to appropriate integer
df['work_year'] = df['work_year'].astype(int)
df['salary_in_usd'] = df['salary_in_usd'].astype(int)

In [19]:
# since in 'remote_ratio' only values 0 and 100 are presented, we can change type to smaller size for efficient data storage
df["remote_flag"] = (df["remote_ratio"] == 100).astype("int8")
df = df.drop(columns=["remote_ratio"])

In [21]:
# converting to ISO-3 standard features containing country abbreviations
df["company_location"] = df["company_location"].apply(to_iso3)
df["employee_residence"] = df["employee_residence"].apply(to_iso3)

In [95]:
# typecasting categorical features
categorical_cols = [
    "experience_level",
    "employment_type",
    "job_title",
    "salary_currency",
    "company_location",
    "company_size"
]

for col in categorical_cols:
    df[col] = df[col].astype("category")

In [23]:
df['employee_residence'] = df['employee_residence'].astype("string")

In [25]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 93597 entries, 0 to 93596
Data columns (total 11 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   work_year           93597 non-null  int32 
 1   experience_level    93597 non-null  object
 2   employment_type     93597 non-null  object
 3   job_title           93597 non-null  object
 4   salary              93597 non-null  int64 
 5   salary_currency     93597 non-null  object
 6   salary_in_usd       93597 non-null  int32 
 7   employee_residence  93595 non-null  string
 8   company_location    93595 non-null  object
 9   company_size        93597 non-null  object
 10  remote_flag         93597 non-null  int8  
dtypes: int32(2), int64(1), int8(1), object(6), string(1)
memory usage: 6.5+ MB


## adding common column for manipulations with other tables (table merges): country_name + year
in the current dataset we take fields 'employee_residence' and 'work_year' to compose it

In [30]:
df["country_year"] = (df["employee_residence"] + df["work_year"].astype(str))

In [32]:
df.head()

,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,company_location,company_size,remote_flag,country_year
0,2025,MI,FT,Research Scientist,208000,USD,208000,USA,USA,M,0,USA2025
1,2025,MI,FT,Research Scientist,147000,USD,147000,USA,USA,M,0,USA2025
2,2025,SE,FT,Research Scientist,173000,USD,173000,USA,USA,M,0,USA2025
3,2025,SE,FT,Research Scientist,117000,USD,117000,USA,USA,M,0,USA2025
4,2025,MI,FT,AI Engineer,100000,USD,100000,USA,USA,M,1,USA2025


In [36]:
df.describe()

,work_year,salary,salary_in_usd,remote_flag
count,93597.000000,9.359700e+04,93597.000000,93597.000000
mean,2024.086434,1.623541e+05,157547.696774,0.212966
std,0.641449,2.221425e+05,73649.113729,0.409406
min,2020.000000,1.400000e+04,15000.000000,0.000000
25%,2024.000000,1.062600e+05,106250.000000,0.000000
50%,2024.000000,1.470000e+05,146232.000000,0.000000
75%,2024.000000,1.990000e+05,198000.000000,0.000000
max,2025.000000,3.040000e+07,800000.000000,1.000000


## Saving preprocessed dataset

In [38]:
df.to_json(
    "processed_data/DataScience_salaries_2025_clean.json",
    orient="records",
    indent=2,
    force_ascii=False
)